In [ ]:
from typing import Tuple, Dict, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    CLIPVisionModel,
    CLIPTextModel,
    CLIPTokenizer,
    CLIPImageProcessor,
    CLIPModel,
)
from transformers.modeling_outputs import BaseModelOutput
import inspect

In [2]:
MODEL_NAME = "openai/clip-vit-base-patch32"

In [3]:
torch.manual_seed(0)

In [4]:
clip_vision_model = CLIPVisionModel.from_pretrained(MODEL_NAME)
clip_text_model = CLIPTextModel.from_pretrained(MODEL_NAME)
tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME)
processor = CLIPImageProcessor.from_pretrained(MODEL_NAME)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.final_layer_norm.weight                           | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_n

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
logit_scale           

In [5]:
clip_vision_model

CLIPVisionModel(
  (embeddings): CLIPVisionEmbeddings(
    (patch_embedding): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (position_embedding): Embedding(50, 768)
  )
  (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (encoder): CLIPEncoder(
    (layers): ModuleList(
      (0-11): 12 x CLIPEncoderLayer(
        (self_attn): CLIPAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): CLIPMLP(
          (activation_fn): QuickGELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
        )
    

In [6]:
clip_vision_model.encoder

CLIPEncoder(
  (layers): ModuleList(
    (0-11): 12 x CLIPEncoderLayer(
      (self_attn): CLIPAttention(
        (k_proj): Linear(in_features=768, out_features=768, bias=True)
        (v_proj): Linear(in_features=768, out_features=768, bias=True)
        (q_proj): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): CLIPMLP(
        (activation_fn): QuickGELUActivation()
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
      )
      (layer_norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    )
  )
)

In [7]:
print(inspect.getsource(clip_vision_model.encoder.forward))

    def forward(
        self,
        inputs_embeds,
        attention_mask: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> BaseModelOutput:
        hidden_states = inputs_embeds
        for encoder_layer in self.layers:
            hidden_states = encoder_layer(
                hidden_states,
                attention_mask,
                **kwargs,
            )

        return BaseModelOutput(
            last_hidden_state=hidden_states,
        )



In [8]:
print(inspect.getsource(clip_vision_model.forward))

    @merge_with_config_defaults
    @capture_outputs(tie_last_hidden_states=False)
    @auto_docstring
    def forward(
        self,
        pixel_values: torch.FloatTensor | None = None,
        interpolate_pos_encoding: bool | None = False,
        **kwargs: Unpack[TransformersKwargs],
    ) -> BaseModelOutputWithPooling:
        r"""
        Example:

        ```python
        >>> from PIL import Image
        >>> import httpx
        >>> from io import BytesIO
        >>> from transformers import AutoProcessor, CLIPVisionModel

        >>> model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
        >>> processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")

        >>> url = "http://images.cocodataset.org/val2017/000000039769.jpg"
        >>> with httpx.stream("GET", url) as response:
        ...     image = Image.open(BytesIO(response.read()))

        >>> inputs = processor(images=image, return_tensors="pt")

        >>> outputs = model(**i

In [9]:
class UniResImageEncoder(nn.Module):
    def __init__(self, image_encoder: CLIPVisionModel) -> None:
        super().__init__()
        self.image_encoder = image_encoder
        d_model = self.image_encoder.config.hidden_size  # 768
        self.low_tokens = nn.Parameter(torch.rand(64, d_model))
        self.high_tokens = nn.Parameter(torch.rand(8, d_model))

    def forward(self, pixel_values: torch.Tensor) -> Dict[str, torch.Tensor]:
        # tokenize the image + positional embedding
        # (B,50,768)
        hidden_states = self.image_encoder.embeddings(pixel_values)
        batch_size, seq_len, _ = hidden_states.shape

        # insert low level group tokens
        # (B,64,768)
        low_tokens = self.low_tokens.unsqueeze(0).expand(batch_size, -1, -1)
        # (B,114,768)
        hidden_states = torch.cat((hidden_states, low_tokens), dim=1)

        # norm
        hidden_states = self.image_encoder.pre_layrnorm(hidden_states)

        # first encoder half
        for encoder_layer in self.image_encoder.encoder.layers[:6]:
            hidden_states = encoder_layer(hidden_states, attention_mask=None)

        # take out low level group tokens
        low_tokens = hidden_states[:, seq_len:, :]
        # (B,50,768)
        hidden_states = hidden_states[:, :seq_len, :]

        # insert high level group tokens
        # (B,8,768)
        high_tokens = self.high_tokens.unsqueeze(0).expand(batch_size, -1, -1)
        # (B,58,768)
        hidden_states = torch.cat((hidden_states, high_tokens), dim=1)

        # second encoder half
        for encoder_layer in self.image_encoder.encoder.layers[6:]:
            hidden_states = encoder_layer(hidden_states, attention_mask=None)

        # take out high level group tokens
        high_tokens = hidden_states[:, seq_len:, :]
        # (B,50,768)
        last_hidden_state = hidden_states[:, :seq_len, :]

        # CLS token
        # (B,768)
        pooled_output = last_hidden_state[:, 0, :]
        # (B,768)
        pooled_output = self.image_encoder.post_layernorm(pooled_output)

        return {
            "last_hidden_state": last_hidden_state,
            "pooler_output": pooled_output,
            "low_tokens": low_tokens,
            "high_tokens": high_tokens,
        }

In [10]:
class LanguageGuidedRegionFilter(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward) -> None:
        super().__init__()

        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Linear(dim_feedforward, d_model),
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(
        self, group_feat: torch.Tensor, text_feat: torch.Tensor
    ) -> torch.Tensor:
        # cross attention
        normed_group = self.norm1(group_feat)
        fused_feat, _ = self.cross_attn(
            query=normed_group, key=text_feat, value=text_feat
        )
        fused_feat = group_feat + fused_feat

        # feed forward network
        normed_fused = self.norm2(fused_feat)
        ffn_out = self.ffn(normed_fused)
        ffn_out = fused_feat + ffn_out

        return ffn_out


In [54]:
class VLDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward) -> None:
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)

        self.norm2 = nn.LayerNorm(d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)

        self.norm3 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Linear(dim_feedforward, d_model),
        )

    def forward(
        self,
        image_feat: torch.Tensor,
        text_feat: torch.Tensor,
    ) -> torch.Tensor:
        # self attention
        normed_image = self.norm1(image_feat)
        image_feat_2, _ = self.self_attn(
            query=normed_image, key=normed_image, value=normed_image
        )
        image_feat_2 = image_feat + image_feat_2

        # cross attention
        normed_image_2 = self.norm2(image_feat_2)
        fused_feat, _ = self.cross_attn(
            query=normed_image_2, key=text_feat, value=text_feat
        )
        fused_feat = image_feat_2 + fused_feat

        # feed forward network
        normed_fused = self.norm3(fused_feat)
        ffn_out = self.ffn(normed_fused)
        ffn_out = fused_feat + ffn_out

        return ffn_out


class VLDecoder1(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers) -> None:
        super().__init__()
        self.layers = nn.ModuleList(
            [VLDecoderLayer(d_model, nhead, dim_feedforward) for _ in range(num_layers)]
        )

    def forward(
        self, image_feat: torch.Tensor, text_feat: torch.Tensor
    ) -> torch.Tensor:
        x = image_feat
        for layer in self.layers:
            x = layer(x, text_feat)

        return x


class VLDecoder2(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers) -> None:
        super().__init__()

        # transformer
        self.layers = nn.ModuleList(
            [VLDecoderLayer(d_model, nhead, dim_feedforward) for _ in range(num_layers)]
        )

        # upsample
        self.decoder = nn.Sequential(
            # (B,512,7,7) to (B,256,14,14)
            nn.ConvTranspose2d(d_model, 256, kernel_size=2, stride=2),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            # (B,256,14,14) to (B,128,28, 28)
            nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # (B,128,28,28) to (B,64,56,56)
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # (B,64,56,56) to (B,32,112,112)
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # (B,32,112,112) to (B,16,224,224)
            nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            # (B,16,224,224) to (B,1,224,224)
            nn.Conv2d(16, 1, kernel_size=1),
        )

    def forward(
        self, image_feat: torch.Tensor, group_feat: torch.Tensor
    ) -> torch.Tensor:
        # transformer blocks
        # (B,50,512)
        x = image_feat
        for layer in self.layers:
            # (B,50,512)
            x = layer(x, group_feat)

        # reshape for cnn
        # (B,49,512)
        x = x[:, 1:, :]
        batch_size, seq_len, d_model = x.shape

        # (B,512,49)
        x = x.transpose(1, 2)

        height = width = int(seq_len**0.5)
        # (B,512,7,7)
        x = x.reshape(batch_size, d_model, height, width)

        # upsample
        # (B,1,224,224)
        x = self.decoder(x)
        # (B,224,224)
        x = x.squeeze()

        return x

In [ ]:
torch.rand(16, 1, 224, 224).squeeze().shape

torch.Size([16, 224, 224])

In [12]:
image_feat = torch.rand(16, 50, 512)
group_feat = torch.rand(16, 26, 512)
VLDecoder1(512, 8, 512 * 4, 6)(image_feat, group_feat).shape

torch.Size([16, 50, 512])

In [55]:
image_feat = torch.rand(16, 50, 512)
group_feat = torch.rand(16, 72, 512)
VLDecoder2(512, 8, 512 * 4, 6)(image_feat, group_feat).shape

torch.Size([16, 224, 224])

In [56]:
class UniRes(nn.Module):
    def __init__(
        self, image_encoder: CLIPVisionModel, text_encoder: CLIPTextModel
    ) -> None:
        super().__init__()

        self.image_encoder = UniResImageEncoder(image_encoder)
        self.text_encoder = text_encoder
        self.image_projection = nn.Linear(768, 512)
        self.text_projection = nn.Linear(512, 512)

        d_model = 512
        nhead = 8
        dim_feedforward = d_model * 4

        self.lrf = LanguageGuidedRegionFilter(d_model, nhead, dim_feedforward)

        self.decoder1 = VLDecoder1(d_model, nhead, dim_feedforward, num_layers=12)
        self.decoder2 = VLDecoder2(d_model, nhead, dim_feedforward, num_layers=12)

    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        # clip image
        image_out = self.image_encoder(pixel_values)
        # (B,50,768)
        image_feat = image_out["last_hidden_state"]
        # (B,64,768)
        low_tokens = image_out["low_tokens"]
        # (B,8,768)
        high_tokens = image_out["high_tokens"]
        # (B,50,512)
        image_feat = self.image_projection(image_feat)
        # (B,64,512)
        low_tokens = self.image_projection(low_tokens)
        # (B,8,512)
        high_tokens = self.image_projection(high_tokens)

        # clip text
        text_out = self.text_encoder(input_ids, attention_mask)
        # (B,S,512)
        text_feat = text_out["last_hidden_state"]
        # (B,S,512)
        text_feat = self.text_projection(text_feat)

        # language guided region filter (LRF)
        # (B,64,512)
        low_feat = self.lrf(low_tokens, text_feat)
        # (B,8,512)
        high_feat = self.lrf(high_tokens, text_feat)
        # (B,72,512)
        group_feat = torch.concat((low_feat, high_feat), dim=1)

        # stage 1 and stage 2 decoder
        # (B,50,512)
        fused_feat = self.decoder1(image_feat, text_feat)
        # (B,224,224)
        mask = self.decoder2(fused_feat, group_feat)

        return mask


In [57]:
model = UniRes(clip_vision_model, clip_text_model)

In [ ]:
low_tokens = torch.rand(16, 64, 512)
high_tokens = torch.rand(16, 8, 512)
torch.concat((low_tokens, high_tokens), dim=1).shape

torch.Size([16, 72, 512])

In [16]:
["i am subaru", "as the strongest curse jogoat"] * 8

['i am subaru',
 'as the strongest curse jogoat',
 'i am subaru',
 'as the strongest curse jogoat',
 'i am subaru',
 'as the strongest curse jogoat',
 'i am subaru',
 'as the strongest curse jogoat',
 'i am subaru',
 'as the strongest curse jogoat',
 'i am subaru',
 'as the strongest curse jogoat',
 'i am subaru',
 'as the strongest curse jogoat',
 'i am subaru',
 'as the strongest curse jogoat']

In [47]:
clip_inputs = tokenizer(
    [
        "i am subaru",
        "as the strongest curse jogoat",
        "This TransformerEncoder layer implements the original architecture",
        "an instance of the TransformerEncoderLayer() class (required)",
        "the number of sub-encoder-layers in the encoder (required).",
        "Pass the input through the encoder layers in turn.",
        "applies a causal mask as mask",
        " Warning: is_causal provides a hint that mask is the causal mask. Providing incorrect hints can result in incorrect execution, including forward and backward compatibility.",
    ]
    * 2,
    padding=True,
    truncation=True,
    return_tensors="pt",
)
clip_inputs["input_ids"].shape

torch.Size([16, 34])

In [58]:
pixel_values = torch.rand(16, 3, 224, 224)
input_ids = clip_inputs["input_ids"]
attention_mask = clip_inputs["attention_mask"]

mask = model(pixel_values, input_ids, attention_mask)

In [59]:
mask.shape

torch.Size([16, 224, 224])

In [20]:
clip_inputs = tokenizer(
    ["i am subaru", "as the strongest curse jogoat"],
    padding=True,
    truncation=True,
    return_tensors="pt",
)
clip_outputs = clip_text_model(**clip_inputs)
clip_outputs

BaseModelOutputWithPooling(last_hidden_state=tensor([[[ 3.3929e-01,  1.1646e-01,  1.0195e-01,  ...,  2.4677e-01,
           5.9064e-01,  1.0130e-01],
         [ 6.9760e-01, -8.3717e-01,  4.3726e-01,  ...,  1.1181e+00,
          -1.4406e-01, -2.9393e-01],
         [-3.7932e-01,  9.1991e-01,  4.3670e-02,  ...,  2.2799e+00,
          -4.8892e-01, -1.1454e+00],
         ...,
         [ 1.1276e+00,  3.9555e-01, -2.3678e-01,  ...,  2.4398e+00,
          -2.3462e-01, -5.6630e-01],
         [ 1.2047e+00,  3.5047e-01, -2.7333e-01,  ...,  2.4482e+00,
          -1.4395e-01, -5.1987e-01],
         [ 1.2211e+00,  3.5429e-01, -2.4436e-01,  ...,  2.4187e+00,
          -1.1318e-01, -4.9294e-01]],

        [[ 3.3929e-01,  1.1646e-01,  1.0195e-01,  ...,  2.4677e-01,
           5.9064e-01,  1.0130e-01],
         [ 8.2823e-01, -1.2106e+00,  6.5737e-02,  ...,  1.4178e+00,
           7.5970e-01,  1.0249e-01],
         [-3.7181e-02, -7.1953e-01,  7.8511e-02,  ...,  2.4186e-01,
          -1.1970e-03, -5.1704e

In [21]:
clip_outputs.pooler_output.shape

torch.Size([2, 512])

In [22]:
from PIL import Image
import httpx
from io import BytesIO

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
with httpx.stream("GET", url) as response:
    image = Image.open(BytesIO(response.read()))

image_inputs = processor(images=[image, image, image], return_tensors="pt")

image_outputs = clip_vision_model(**image_inputs)

In [23]:
yo = clip_vision_model.embeddings(image_inputs["pixel_values"])

In [24]:
yo.shape

torch.Size([3, 50, 768])

In [25]:
jogoat = nn.Parameter(torch.rand(8, 768))
jogoat.shape

torch.Size([8, 768])

In [26]:
goat = jogoat.unsqueeze(0).expand(3, -1, -1)
goat.shape

torch.Size([3, 8, 768])

In [27]:
torch.cat((yo, goat), dim=1).shape

torch.Size([3, 58, 768])

In [28]:
image_outputs.pooler_output.shape

torch.Size([3, 768])

In [29]:
clip_text_model

CLIPTextModel(
  (embeddings): CLIPTextEmbeddings(
    (token_embedding): Embedding(49408, 512)
    (position_embedding): Embedding(77, 512)
  )
  (encoder): CLIPEncoder(
    (layers): ModuleList(
      (0-11): 12 x CLIPEncoderLayer(
        (self_attn): CLIPAttention(
          (k_proj): Linear(in_features=512, out_features=512, bias=True)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (mlp): CLIPMLP(
          (activation_fn): QuickGELUActivation()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
        )
        (layer_norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (final_layer_norm): LayerNo

In [30]:
clip_vision_model

CLIPVisionModel(
  (embeddings): CLIPVisionEmbeddings(
    (patch_embedding): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (position_embedding): Embedding(50, 768)
  )
  (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (encoder): CLIPEncoder(
    (layers): ModuleList(
      (0-11): 12 x CLIPEncoderLayer(
        (self_attn): CLIPAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): CLIPMLP(
          (activation_fn): QuickGELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
        )
    

In [31]:
import torch
from transformers import AutoProcessor, CLIPModel
from transformers.image_utils import load_image

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = load_image(url)

clip_inputs = processor(
    text=["a photo of a cat", "a photo of a dog"],
    images=image,
    return_tensors="pt",
    padding=True,
)

with torch.inference_mode():
    clip_outputs = model(**clip_inputs)
logits_per_image = (
    clip_outputs.logits_per_image
)  # this is the image-text similarity score
probs = logits_per_image.softmax(
    dim=1
)  # we can take the softmax to get the label probabilities

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [32]:
for key in clip_outputs.keys():
    print(key)

logits_per_image
logits_per_text
text_embeds
image_embeds
text_model_output
vision_model_output


In [33]:
clip_outputs.image_embeds.shape

torch.Size([1, 512])

In [34]:
model

CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05, eleme

In [35]:
clip_vision_model

CLIPVisionModel(
  (embeddings): CLIPVisionEmbeddings(
    (patch_embedding): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (position_embedding): Embedding(50, 768)
  )
  (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (encoder): CLIPEncoder(
    (layers): ModuleList(
      (0-11): 12 x CLIPEncoderLayer(
        (self_attn): CLIPAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): CLIPMLP(
          (activation_fn): QuickGELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
        )
    

In [36]:
uni_image_encoder = UniResImageEncoder(clip_vision_model)

In [37]:
uni_image_encoder

UniResImageEncoder(
  (image_encoder): CLIPVisionModel(
    (embeddings): CLIPVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
      (position_embedding): Embedding(50, 768)
    )
    (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
          

In [38]:
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
with httpx.stream("GET", url) as response:
    image = Image.open(BytesIO(response.read()))

image_inputs = processor(images=[image, image, image], return_tensors="pt")

In [39]:
pixel_values = image_inputs["pixel_values"]
outputs = uni_image_encoder(pixel_values)

In [40]:
for k, v in outputs.items():
    print(f"{k}: {v.shape}")

last_hidden_state: torch.Size([3, 50, 768])
pooler_output: torch.Size([3, 768])
low_tokens: torch.Size([3, 64, 768])
high_tokens: torch.Size([3, 8, 768])


In [41]:
image_inputs["pixel_values"].shape

torch.Size([3, 3, 224, 224])

In [42]:
lrf = LanguageGuidedRegionFilter(512, 8, 512 * 4)
group_tensor = torch.rand(16, 64, 512)
text_tensor = torch.rand(16, 30, 512)
lrf(group_tensor, text_tensor).shape

torch.Size([16, 64, 512])